## 1. Importación de librerías y conexión a MySQL

Importamos las librerías necesarias para conectar Python con MySQL 
y crear una base de datos para el proyecto.

In [16]:
import pandas as pd
import pymysql
from sqlalchemy import create_engine

## 1. Importación de librerías y conexión a MySQL

Importamos pandas, pymysql y sqlalchemy, y creamos la conexión 
a nuestra instancia local de MySQL (root@localhost:3306).

In [17]:
import pandas as pd
import pymysql
from sqlalchemy import create_engine
import getpass

usuario = 'root'
contraseña = getpass.getpass('Introduce tu contraseña de MySQL: ')
host = 'localhost'
puerto = '3306'

engine = create_engine(f'mysql+pymysql://{usuario}:{contraseña}@{host}:{puerto}')

print('Conexión creada')

Conexión creada


**Conclusiones:**

- Conexión a MySQL establecida correctamente desde Python, usando 
  SQLAlchemy + pymysql
- La contraseña se introdujo de forma segura con getpass, sin quedar 
  escrita en el notebook
- El siguiente paso es crear una base de datos para el proyecto 
  y cargar en ella el dataset limpio

## 2. Creación de la base de datos del proyecto

Creamos una base de datos nueva en MySQL llamada "dashboard2_pantallas" 
para almacenar los datos del proyecto.

In [18]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS dashboard2_pantallas"))

print('Base de datos creada (o ya existía)')

Base de datos creada (o ya existía)


**Conclusiones:**

- Base de datos `dashboard2_pantallas` creada correctamente en MySQL
- Detalle técnico: en SQLAlchemy 2.x, los comandos SQL en texto plano 
  deben envolverse con text() para poder ejecutarse
- Esta base de datos es visible también desde MySQL Workbench 
  (misma instancia, mismo servidor)

## 3. Carga del CSV limpio en una tabla de MySQL

Leemos dataset_limpio_dashboard2.csv y lo cargamos como una tabla 
llamada "screen_time" dentro de la base de datos dashboard2_pantallas.

In [19]:
df = pd.read_csv('Datasets/dataset_limpio_dashboard2.csv')

# Conexión específica a la base de datos ya creada
engine_db = create_engine(f'mysql+pymysql://{usuario}:{contraseña}@{host}:{puerto}/dashboard2_pantallas')

df.to_sql('screen_time', engine_db, if_exists='replace', index=False)

print('Tabla screen_time creada con', len(df), 'filas')

Tabla screen_time creada con 3872 filas


**Conclusiones:**

- Tabla `screen_time` creada correctamente en la base de datos 
  `dashboard2_pantallas`, con 3.872 filas (las mismas del CSV limpio)
- La tabla incluye las 10 columnas: variables numéricas limpias 
  (edad, sexo, horas pantalla, PCIAT_Total, CGAS_Score, SDS_Total_T, sii) 
  y las columnas añadidas (Horas_Pantalla_Label, Sexo_Label, Uso_Saludable)
- Ya se puede consultar esta tabla con SQL, tanto desde Python 
  como desde MySQL Workbench

## 4. Primera consulta: vista general de la tabla

Comprobamos que la tabla se cargó correctamente consultando 
las primeras filas directamente con SQL.

In [20]:
query = "SELECT * FROM screen_time LIMIT 5"
pd.read_sql(query, engine_db)

,id,Basic_Demos-Age,Basic_Demos-Sex,PreInt_EduHx-computerinternet_hoursday,PCIAT-PCIAT_Total,CGAS-CGAS_Score,SDS-SDS_Total_T,sii,Horas_Pantalla_Label,Sexo_Label,Uso_Saludable
0,00008ff9,5,0,3,55,51,55,2,Más de 3h/día,Masculino,0
1,000fd460,9,0,0,0,65,64,0,Menos de 1h/día,Masculino,0
2,00105258,10,1,2,28,71,54,0,Alrededor de 2h/día,Femenino,0
3,00115b9f,9,0,0,44,71,45,1,Menos de 1h/día,Masculino,1
4,0016bb22,18,1,1,26,65,55,0,Alrededor de 1h/día,Femenino,0


**Conclusiones:**

- La consulta SQL funciona correctamente desde Python
- Los datos coinciden exactamente con el CSV limpio: 11 columnas, 
  mismos valores para las primeras 5 filas
- Nota: la columna Uso_Saludable se almacenó como 0/1 (booleano 
  convertido a entero por MySQL), en lugar de True/False
- La tabla está lista para hacer consultas más complejas con SQL

## 5. Consulta agregada: PCIAT_Total medio por categoría de horas de pantalla

Usamos GROUP BY para calcular la media de PCIAT_Total y el número 
de participantes en cada categoría de horas de pantalla, 
replicando con SQL lo que vimos antes con gráficos en Python.

In [21]:
query = """
SELECT 
    Horas_Pantalla_Label,
    COUNT(*) AS num_participantes,
    ROUND(AVG(`PCIAT-PCIAT_Total`), 2) AS pciat_promedio
FROM screen_time
GROUP BY Horas_Pantalla_Label
ORDER BY pciat_promedio DESC
"""

pd.read_sql(query, engine_db)

,Horas_Pantalla_Label,num_participantes,pciat_promedio
0,Más de 3h/día,330,39.01
1,Alrededor de 2h/día,985,31.67
2,Alrededor de 1h/día,1037,28.12
3,Menos de 1h/día,1520,21.27


**Conclusiones:**

- Esta consulta SQL confirma con números exactos la tendencia que ya 
  habíamos visto visualmente en el paso 21 del EDA (boxplots):
  - Menos de 1h/día: 1.520 participantes, PCIAT promedio = 21.27
  - Alrededor de 1h/día: 1.037 participantes, PCIAT promedio = 28.12
  - Alrededor de 2h/día: 985 participantes, PCIAT promedio = 31.67
  - Más de 3h/día: 330 participantes, PCIAT promedio = 39.01

- Relación clara y monótona: a más horas de pantalla, mayor PCIAT_Total 
  promedio. La diferencia entre el grupo de menos uso (21.27) y el de 
  mayor uso (39.01) es de casi 18 puntos, casi el doble

- Esta tabla resumen es perfecta para Tableau: se puede crear directamente 
  un gráfico de barras "PCIAT promedio por categoría de horas de pantalla" 
  con estos 4 valores

## 6. Consulta: distribución de sii según Uso_Saludable

Comparamos el nivel de severidad (sii) entre el grupo "Uso_Saludable" 
y el resto, para validar si esta variable creada tiene sentido 
(esperamos que el grupo saludable tenga mayoría de sii=0).

In [22]:
query = """
SELECT 
    Uso_Saludable,
    sii,
    COUNT(*) AS num_participantes
FROM screen_time
GROUP BY Uso_Saludable, sii
ORDER BY Uso_Saludable, sii
"""

pd.read_sql(query, engine_db)

,Uso_Saludable,sii,num_participantes
0,0,0,2393
1,0,1,609
2,0,2,338
3,0,3,32
4,1,0,357
5,1,1,109
6,1,2,34


**Conclusiones:**

- Grupo Uso_Saludable=0 (no cumple criterio, 3.372 participantes):
  - sii=0: 2.393 (71.0%)
  - sii=1: 609 (18.1%)
  - sii=2: 338 (10.0%)
  - sii=3: 32 (0.9%)

- Grupo Uso_Saludable=1 (cumple criterio, 500 participantes):
  - sii=0: 357 (71.4%)
  - sii=1: 109 (21.8%)
  - sii=2: 34 (6.8%)
  - sii=3: 0 (0%)

- Hallazgo interesante: el porcentaje de sii=0 es prácticamente igual 
  en ambos grupos (~71%), pero el grupo "Uso_Saludable" tiene 0 casos 
  de sii=3 (severo), mientras que el otro grupo tiene 32 casos (0.9%)

- Esto sugiere que "Uso_Saludable" (definido como ≤1h pantalla + CGAS>70) 
  no predice tanto el nivel general de severidad, pero sí parece asociarse 
  con la ausencia de casos extremos (sii=3)

- Limitación a mencionar: el grupo Uso_Saludable es pequeño (500 vs 3.372), 
  por lo que la ausencia de sii=3 podría deberse en parte al tamaño 
  reducido de la muestra, no solo al efecto protector

## 7. sii promedio por grupo de edad

Creamos grupos de edad (infancia/preadolescencia/adolescencia) 
y calculamos el sii promedio en cada grupo, replicando con SQL 
la tendencia observada en el paso 26 del EDA.

In [23]:
query = """
SELECT 
    CASE 
        WHEN `Basic_Demos-Age` BETWEEN 5 AND 9 THEN 'Infancia (5-9)'
        WHEN `Basic_Demos-Age` BETWEEN 10 AND 13 THEN 'Preadolescencia (10-13)'
        ELSE 'Adolescencia (14-18)'
    END AS grupo_edad,
    COUNT(*) AS num_participantes,
    ROUND(AVG(sii), 2) AS sii_promedio
FROM screen_time
GROUP BY grupo_edad
ORDER BY sii_promedio DESC
"""

pd.read_sql(query, engine_db)

,grupo_edad,num_participantes,sii_promedio
0,Adolescencia (14-18),717,0.67
1,Preadolescencia (10-13),1281,0.51
2,Infancia (5-9),1874,0.22


**Conclusiones:**

- Confirmación con SQL de la tendencia vista en el paso 26 del EDA: 
  el sii promedio aumenta claramente con la edad
  - Infancia (5-9 años): 1.874 participantes, sii promedio = 0.22
  - Preadolescencia (10-13 años): 1.281 participantes, sii promedio = 0.51
  - Adolescencia (14-18 años): 717 participantes, sii promedio = 0.67

- El sii promedio en adolescencia es **3 veces mayor** que en la infancia 
  (0.67 vs 0.22), un salto muy significativo

- La infancia es el grupo más numeroso (1.874, ~48% del dataset), 
  coherente con la distribución de edades sesgada hacia jóvenes 
  vista en el paso 19

- Para Tableau: esta tabla resume perfectamente el mensaje "el riesgo 
  de uso problemático de internet aumenta con la edad", ideal para 
  un gráfico de barras simple

## 8. Distribución de sii por sexo (porcentajes)

Calculamos, para cada sexo, el porcentaje de participantes 
en cada nivel de sii, replicando con SQL el paso 28 del EDA.

In [24]:
query = """
SELECT 
    Sexo_Label,
    sii,
    COUNT(*) AS num_participantes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY Sexo_Label), 2) AS porcentaje
FROM screen_time
GROUP BY Sexo_Label, sii
ORDER BY Sexo_Label, sii
"""

pd.read_sql(query, engine_db)

,Sexo_Label,sii,num_participantes,porcentaje
0,Femenino,0,1082,75.56
1,Femenino,1,244,17.04
2,Femenino,2,99,6.91
3,Femenino,3,7,0.49
4,Masculino,0,1668,68.36
5,Masculino,1,474,19.43
6,Masculino,2,273,11.19
7,Masculino,3,25,1.02


**Conclusiones:**

- Confirmación con SQL de los porcentajes vistos en el paso 28 del EDA, 
  ahora calculados directamente con función de ventana (OVER PARTITION BY):
  - Femenino: sii=0 → 75.56%, sii=1 → 17.04%, sii=2 → 6.91%, sii=3 → 0.49%
  - Masculino: sii=0 → 68.36%, sii=1 → 19.43%, sii=2 → 11.19%, sii=3 → 1.02%

- Se confirma que los hombres tienen mayor proporción en los niveles 
  de severidad moderado y severo (11.19% + 1.02% = 12.21%) que las 
  mujeres (6.91% + 0.49% = 7.40%)

- La función de ventana SQL permite calcular porcentajes "dentro de grupo" 
  en una sola consulta, sin necesidad de hacer cálculos adicionales en Python

## 9. Cruce de horas de pantalla, sexo y sii promedio

Combinamos las tres variables (horas de pantalla, sexo, sii) 
para ver si la relación entre horas y severidad es distinta 
entre hombres y mujeres.

In [25]:
query = """
SELECT 
    Horas_Pantalla_Label,
    Sexo_Label,
    COUNT(*) AS num_participantes,
    ROUND(AVG(sii), 2) AS sii_promedio
FROM screen_time
GROUP BY Horas_Pantalla_Label, Sexo_Label
ORDER BY Horas_Pantalla_Label, Sexo_Label
"""

pd.read_sql(query, engine_db)

,Horas_Pantalla_Label,Sexo_Label,num_participantes,sii_promedio
0,Alrededor de 1h/día,Femenino,391,0.21
1,Alrededor de 1h/día,Masculino,646,0.30
2,Alrededor de 2h/día,Femenino,345,0.54
3,Alrededor de 2h/día,Masculino,640,0.61
4,Más de 3h/día,Femenino,127,0.68
5,Más de 3h/día,Masculino,203,0.98
6,Menos de 1h/día,Femenino,569,0.19
7,Menos de 1h/día,Masculino,951,0.33


**Conclusiones:**

- En ambos sexos se mantiene la tendencia general: a más horas de 
  pantalla, mayor sii promedio

- La diferencia entre sexos se AMPLÍA a medida que aumentan las horas:
  - Menos de 1h/día: Femenino=0.19, Masculino=0.33 (diferencia 0.14)
  - Alrededor de 1h/día: Femenino=0.21, Masculino=0.30 (diferencia 0.09)
  - Alrededor de 2h/día: Femenino=0.54, Masculino=0.61 (diferencia 0.07)
  - Más de 3h/día: Femenino=0.68, Masculino=0.98 (diferencia 0.30)

- El hallazgo más destacado: en la categoría "Más de 3h/día", los hombres 
  tienen un sii promedio de 0.98 (casi 1, "leve"), mientras que las mujeres 
  se quedan en 0.68 — la brecha entre sexos es mayor precisamente 
  en el grupo de mayor uso

- Para Tableau: este cruce de 3 variables podría visualizarse como 
  un gráfico de líneas o barras agrupadas, mostrando ambas líneas 
  (Femenino/Masculino) a lo largo de las 4 categorías de horas, 
  donde se aprecia cómo "se separan" en el extremo de mayor uso

## 10. Resumen estadístico general (tipo describe() en SQL)

Calculamos min, max, media y desviación estándar de las variables 
numéricas principales directamente con SQL.

In [26]:
query = """
SELECT 
    'Basic_Demos-Age' AS variable,
    MIN(`Basic_Demos-Age`) AS minimo,
    MAX(`Basic_Demos-Age`) AS maximo,
    ROUND(AVG(`Basic_Demos-Age`), 2) AS media,
    ROUND(STDDEV(`Basic_Demos-Age`), 2) AS desviacion
FROM screen_time

UNION ALL

SELECT 
    'PCIAT-PCIAT_Total',
    MIN(`PCIAT-PCIAT_Total`),
    MAX(`PCIAT-PCIAT_Total`),
    ROUND(AVG(`PCIAT-PCIAT_Total`), 2),
    ROUND(STDDEV(`PCIAT-PCIAT_Total`), 2)
FROM screen_time

UNION ALL

SELECT 
    'CGAS-CGAS_Score',
    MIN(`CGAS-CGAS_Score`),
    MAX(`CGAS-CGAS_Score`),
    ROUND(AVG(`CGAS-CGAS_Score`), 2),
    ROUND(STDDEV(`CGAS-CGAS_Score`), 2)
FROM screen_time

UNION ALL

SELECT 
    'SDS-SDS_Total_T',
    MIN(`SDS-SDS_Total_T`),
    MAX(`SDS-SDS_Total_T`),
    ROUND(AVG(`SDS-SDS_Total_T`), 2),
    ROUND(STDDEV(`SDS-SDS_Total_T`), 2)
FROM screen_time
"""

pd.read_sql(query, engine_db)

,variable,minimo,maximo,media,desviacion
0,Basic_Demos-Age,5,18,10.21,3.29
1,PCIAT-PCIAT_Total,0,93,27.26,16.95
2,CGAS-CGAS_Score,25,95,65.10,9.22
3,SDS-SDS_Total_T,38,100,56.83,10.84


**Conclusiones:**

- Resumen estadístico consistente con lo visto en el paso 11 del EDA 
  (describe() en Python), confirmando que la carga a MySQL no alteró 
  los datos:
  - Edad: rango 5-18, media 10.21, desviación 3.29
  - PCIAT_Total: rango 0-93, media 27.26, desviación 16.95
  - CGAS_Score: rango 25-95, media 65.10, desviación 9.22 (outlier 999 
    ya corregido, máximo correcto en 95)
  - SDS_Total_T: rango 38-100, media 56.83, desviación 10.84

- Esta consulta demuestra cómo obtener estadísticas descriptivas 
  directamente en SQL mediante UNION ALL, útil si se necesita 
  un resumen "tipo describe()" sin pasar por pandas

## 11. Comparativa de perfiles: sii=0 (sin problema) vs sii=3 (severo)

Comparamos los valores promedio de edad, horas de pantalla, sueño 
y funcionamiento general entre el grupo sin problema (sii=0) 
y el grupo severo (sii=3), para responder "¿qué perfil tiene mayor riesgo?"

In [27]:
query = """
SELECT 
    sii,
    COUNT(*) AS num_participantes,
    ROUND(AVG(`Basic_Demos-Age`), 2) AS edad_promedio,
    ROUND(AVG(`PreInt_EduHx-computerinternet_hoursday`), 2) AS horas_pantalla_promedio,
    ROUND(AVG(`SDS-SDS_Total_T`), 2) AS sueño_promedio,
    ROUND(AVG(`CGAS-CGAS_Score`), 2) AS cgas_promedio
FROM screen_time
WHERE sii IN (0, 3)
GROUP BY sii
"""

pd.read_sql(query, engine_db)

,sii,num_participantes,edad_promedio,horas_pantalla_promedio,sueño_promedio,cgas_promedio
0,0,2750,9.70,0.89,55.40,65.39
1,3,32,14.34,2.06,68.69,59.66


**Conclusiones:**

- Comparativa de perfiles entre el grupo "sin problema" (sii=0, 2.750 
  participantes) y el grupo "severo" (sii=3, solo 32 participantes):

  - **Edad**: el grupo severo es notablemente mayor (14.34 vs 9.70 años, 
    +4.6 años de diferencia) → confirma que la adolescencia concentra 
    el mayor riesgo

  - **Horas de pantalla**: el grupo severo usa más pantallas en promedio 
    (2.06 vs 0.89, más del doble) → categoría "alrededor de 2h/día" 
    vs "menos de 1h/día"

  - **Sueño (SDS_Total_T)**: el grupo severo tiene peores indicadores 
    de sueño (68.69 vs 55.40, +13.3 puntos) → a mayor severidad, 
    peor calidad de sueño, una relación que no se veía tan clara 
    en el análisis general (paso 22)

  - **CGAS_Score**: el grupo severo tiene peor funcionamiento general 
    (59.66 vs 65.39, -5.7 puntos), aunque la diferencia es más moderada 
    que en sueño

- **Perfil de mayor riesgo (sii=3)**: adolescente (~14 años), con uso 
  de pantalla alto (~2h/día), problemas de sueño notables y 
  funcionamiento general algo reducido

- ⚠️ Limitación: el grupo sii=3 tiene solo 32 casos, por lo que estas 
  medias son menos robustas estadísticamente que las del grupo sii=0 
  (2.750 casos). Aun así, la dirección de todas las diferencias es 
  coherente con el resto del análisis

## 12. Porcentaje de "Uso_Saludable" por categoría de horas de pantalla

Calculamos qué porcentaje de cada categoría de horas de pantalla 
cumple el criterio "Uso_Saludable" (≤1h + CGAS>70), para ver si 
esta variable positiva se concentra en las categorías de menor uso.

In [28]:
query = """
SELECT 
    Horas_Pantalla_Label,
    COUNT(*) AS total,
    SUM(Uso_Saludable) AS num_uso_saludable,
    ROUND(SUM(Uso_Saludable) * 100.0 / COUNT(*), 2) AS porcentaje_uso_saludable
FROM screen_time
GROUP BY Horas_Pantalla_Label
ORDER BY porcentaje_uso_saludable DESC
"""

pd.read_sql(query, engine_db)

,Horas_Pantalla_Label,total,num_uso_saludable,porcentaje_uso_saludable
0,Menos de 1h/día,1520,386.0,25.39
1,Alrededor de 1h/día,1037,114.0,10.99
2,Más de 3h/día,330,0.0,0.00
3,Alrededor de 2h/día,985,0.0,0.00


**Conclusiones:**

- El "Uso_Saludable" se concentra casi exclusivamente en las categorías 
  de menor uso de pantalla:
  - Menos de 1h/día: 25.39% cumple el criterio (386 de 1.520)
  - Alrededor de 1h/día: 10.99% cumple el criterio (114 de 1.037)
  - Alrededor de 2h/día: 0% (0 de 985)
  - Más de 3h/día: 0% (0 de 330)

- Resultado esperado por construcción: recordemos que Uso_Saludable 
  se definió como (horas ≤1h/día) AND (CGAS>70) — por definición, 
  las categorías 2 y 3 (≥2h/día) NO pueden cumplir el criterio, 
  de ahí el 0% exacto

- El dato más informativo es la diferencia DENTRO de las categorías 
  que sí pueden cumplirlo: en "menos de 1h/día" el porcentaje (25.39%) 
  es más del doble que en "alrededor de 1h/día" (10.99%) — sugiere que 
  cuanto menos tiempo de pantalla, más probable es también tener 
  buen funcionamiento general (CGAS>70)

- Para Tableau: este gráfico de barras (4 categorías, 2 con 0%) 
  visualiza claramente el mensaje "el uso saludable, tal como lo 
  definimos, solo aparece con poco tiempo de pantalla"

## 13. Creación de una VIEW resumen para Tableau

Creamos una vista (VIEW) en MySQL que combina las columnas más 
relevantes, lista para conectar directamente desde Tableau 
sin necesidad de repetir consultas.

In [29]:
query_view = """
CREATE OR REPLACE VIEW vista_dashboard2 AS
SELECT 
    id,
    `Basic_Demos-Age` AS edad,
    Sexo_Label AS sexo,
    Horas_Pantalla_Label AS horas_pantalla,
    `PreInt_EduHx-computerinternet_hoursday` AS horas_pantalla_codigo,
    `PCIAT-PCIAT_Total` AS pciat_total,
    `CGAS-CGAS_Score` AS cgas_score,
    `SDS-SDS_Total_T` AS sueño_score,
    sii,
    Uso_Saludable AS uso_saludable
FROM screen_time
"""

with engine_db.connect() as conn:
    conn.execute(text(query_view))

print('Vista vista_dashboard2 creada correctamente')

Vista vista_dashboard2 creada correctamente


**Conclusiones:**

- Vista `vista_dashboard2` creada correctamente en la base de datos 
  `dashboard2_pantallas`
- Renombra las columnas técnicas (con guiones y prefijos) a nombres 
  limpios en español: edad, sexo, horas_pantalla, pciat_total, 
  cgas_score, sueño_score, sii, uso_saludable
- Esta vista es la que se conectará desde Tableau para construir 
  el Dashboard 2, evitando problemas con nombres de columna 
  complejos (guiones, mayúsculas inconsistentes)